# Spark Core API by Examples

This notebook illustrates how the spark CORE api works by means of simple examples. The notebook is executed in python through pyspark (and jupyter).

The section "Installing spark locally" included below illustrates how to install pyspark and *must* be executed every time you are running this notebook in a Google Colab runtime. (Such runtimes do not have spark installed by default). If you are running this notebook on your local computer, it is recommended to install spark once and for all, instead of executing it for every notebook that you wish to use spark in.

## Attention!
This notebook contains some linux-specific commands to download datasets. If you are running on windows, it is advised to run this notebook through Google Colab.

### Useful documentation.

The PySpark Documentation is available at https://spark.apache.org/docs/latest/api/python/index.html. 

Instructions on how to install PySpark on your local PC may be found at https://spark.apache.org/docs/latest/api/python/getting_started/install.html. Note that by installing PySpark in this way, you automatically have a local copy of Spark.

The Spark Core  api that we use below has the following  documentation which is a useful reference to have: https://spark.apache.org/docs/latest/api/python/reference/pyspark.html

#### Installing PySpark

In [1]:
# This installs pyspark in the current python environment.
# By installing pyspark, we automatically also install spark.
!pip install pyspark

#### General imports and starting Spark

In [2]:
#This is needed to start a Spark session from the notebook
#You may adjust the memory used by the driver program based on your machine's settings
import os 
os.environ['PYSPARK_SUBMIT_ARGS'] ="--conf spark.driver.memory=3g  pyspark-shell"

from pyspark.sql import SparkSession

In [3]:
import os
import sys
os.chdir(r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment")
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# -------------------------------
# Start Spark in LOCAL mode
# -------------------------------

#The following lines are just there to allow this cell to be re-executed multiple times:
#if a spark session was already started, we stop it before starting a new one
#(there can be only one spark context per jupyter notebook)
try: 
    spark
    print("Spark application already started. Terminating existing application and starting new one")
    spark.stop()
except: 
    pass

# Create a new spark session (note, the * indicates to use all available CPU cores)
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .appName("demoRDD") \
    .getOrCreate()
    
#When dealing with RDDs, we work the sparkContext object. See https://spark.apache.org/docs/latest/api/python/pyspark.html#pyspark.SparkContext
sc=spark.sparkContext

# We print the sparkcontext. This prints general information about the spark instance we have connected to. 
# In particular, the hyperlink allows us to open the spark UI (useful for seeing what is going on)
# Note: this hyperlink won't work when running this notebook in Google Colab.
sc

C:\Users\gihame\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


<SparkContext master=local[2] appName=demoRDD>

In [4]:
# check that we have a working spark context, print its configuration
sc._conf.getAll()

[('spark.rdd.compress', 'True'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.app.startTime', '1790893932999'),
 ('spark.sql.artifact.isolation.enabled', 'false'),
 ('spark.driver.port', '54250'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED -

### Downloading data

In [5]:
# Local data is already in this folder. wget/unzip/ls are Linux-only.
import os
root = r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment"
os.chdir(root)
print("Working directory:", os.getcwd())
for name in ["books", "sensors", "movielens", "github"]:
    folder = os.path.join("data", name)
    print(f"{folder}: {len(os.listdir(folder))} files")

Working directory: C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment
data\books: 30 files
data\sensors: 1 files
data\movielens: 5 files
data\github: 2 files


### Word Count in Spark

The `data/books` folder contains books (in txt format) downloaded from project gutenberg (http://www.gutenberg.org/cache/epub/). In that folder you will also find a shell script to download more books (`data/books/download_more_books.sh`), should you wish to experiment with a larger collection of files.

In [6]:
import os
print("\n".join(os.listdir("data/books")))

download_more_books.sh
pg20417.txt
pg20418.txt
pg20419.txt
pg20420.txt
pg20421.txt
pg20422.txt
pg20423.txt
pg20424.txt
pg20425.txt
pg20426.txt
pg20427.txt
pg20428.txt
pg20429.txt
pg20430.txt
pg20431.txt
pg20432.txt
pg20433.txt
pg20434.txt
pg20435.txt
pg20436.txt
pg20437.txt
pg20438.txt
pg20439.txt
pg20440.txt
pg20441.txt
pg20442.txt
pg20443.txt
pg20444.txt
pg20445.txt


In [7]:
# Load the contents of a file into an RDD. Note - when run on the cluster this load from HDFS (inside /user/$USER/)
# if you really want to load from HDFS, you can also put the full HDFS url, e.g.
# hdfs://public00:8020/user/<your_user_id_here>/data/books/pg20417.txt
fileName = "file:///C:/Users/gihame/Documents/UHasselt/First Year/Advanced Topic in DS/spark-exercise-assignment/data/books/pg20417.txt"
bookRDD = sc.textFile(fileName)

In [8]:
# Show 5 elements of bookRDD
bookRDD.take(5)
#bookRDD.glom().first()[:5]
# When loading a textfile, it is hence converted to an RDD where each line is one element. 

['The Project Gutenberg EBook of The Outline of Science, Vol. 1 (of 4), by ',
 'J. Arthur Thomson',
 '',
 'This eBook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever.  You may copy it, give it away or']

In [9]:
# We can operate on each element of an RDD by invoking the map() transformation.
#
# MAP : Return a new RDD by applying a function to each element of this RDD.
# 
# Specifically, we split each line in multiples values by splitting on whitespace separators
# The result is hence an RDD where each element is itself a list.
tupleRDD = bookRDD.map(lambda line: line.split())
tupleRDD.take(3)

[['The',
  'Project',
  'Gutenberg',
  'EBook',
  'of',
  'The',
  'Outline',
  'of',
  'Science,',
  'Vol.',
  '1',
  '(of',
  '4),',
  'by'],
 ['J.', 'Arthur', 'Thomson'],
 []]

In [10]:
# Let's turn the bookRDD into an RDD of words - each word becomes an element in the RDD
# To do so, we use flatMap instead of map
#
# FLATMAP : Return a new RDD by first applying a function to all elements of this RDD, 
#           and then flattening the results.
# 
wordsRDD = bookRDD.flatMap(lambda line: line.split())
wordsRDD.take(5)

['The', 'Project', 'Gutenberg', 'EBook', 'of']

In [11]:
# do a wordCount by invoking the count() action
wordsRDD.count()

109844

In [12]:
# Alternatively, we can get the same result by using the reduce() action. 
#
# REDUCE : Reduces the elements of this RDD into a single value using the specified commutative and 
#          assocative binary operator. Currently reduces partitions locally.
# Note: we first map the RDD into an RDD of integers
wordsRDD.map(lambda x: 1).reduce(lambda a,b: a+b)

109844

**Exercise**: what happens if you reduce wordsRDD without mapping it first to an RDD of integers?

In [13]:
# Your answer here: reduce wordsRDD without mapping it first to an RDD of integers. 
# Can you explain why this happens ?
wordsRDD.reduce(lambda a,b: a+b)

'TheProjectGutenbergEBookofTheOutlineofScience,Vol.1(of4),byJ.ArthurThomsonThiseBookisfortheuseofanyoneanywhereatnocostandwithalmostnorestrictionswhatsoever.Youmaycopyit,giveitawayorre-useitunderthetermsoftheProjectGutenbergLicenseincludedwiththiseBookoronlineatwww.gutenberg.orgTitle:TheOutlineofScience,Vol.1(of4)APlainStorySimplyToldAuthor:J.ArthurThomsonReleaseDate:January22,2007[EBook#20417]Language:English***STARTOFTHISPROJECTGUTENBERGEBOOKOUTLINEOFSCIENCE***ProducedbyBrianJanes,LeonardJohnsonandtheOnlineDistributedProofreadingTeamathttp://www.pgdp.net[Illustration:THEGREATSCARLETSOLARPROMINENCES,WHICHARESUCHANOTABLEFEATUREOFTHESOLARPHENOMENA,AREIMMENSEOUTBURSTSOFFLAMINGHYDROGENRISINGSOMETIMESTOAHEIGHTOF500,000MILES]THEOUTLINEOFSCIENCEAPLAINSTORYSIMPLYTOLDEDITEDBYJ.ARTHURTHOMSONREGIUSPROFESSOROFNATURALHISTORYINTHEUNIVERSITYOFABERDEENWITHOVER800ILLUSTRATIONSOFWHICHABOUT40AREINCOLOURINFOURVOLUMESG.P.PUTNAM\'SSONSNEWYORKANDLONDONTheKnickerbockerpressCopyright,1922byG.P.Putnam\'sSons_F

In [14]:
# count how many unique words there are. The distinct() transformation removes duplicate elements.
wordsRDD.distinct().count()

18039

In [15]:
# Count how many times each unique word appears in wordsRDD by invoking the countByValue() transformation
wordsRDD.countByValue()

defaultdict(int,
            {'The': 876,
             'Project': 78,
             'Gutenberg': 21,
             'EBook': 2,
             'of': 5425,
             'Outline': 7,
             'Science,': 3,
             'Vol.': 4,
             '1': 24,
             '(of': 3,
             '4),': 2,
             'by': 635,
             'J.': 95,
             'Arthur': 9,
             'Thomson': 6,
             'This': 133,
             'eBook': 5,
             'is': 2068,
             'for': 522,
             'the': 7906,
             'use': 43,
             'anyone': 6,
             'anywhere': 3,
             'at': 390,
             'no': 215,
             'cost': 3,
             'and': 2759,
             'with': 584,
             'almost': 59,
             'restrictions': 2,
             'whatsoever.': 2,
             'You': 15,
             'may': 289,
             'copy': 8,
             'it,': 30,
             'give': 50,
             'it': 912,
             'away': 37,
             

### Do a wordcount on all files

In [16]:
import glob, os
book_files = sorted(glob.glob(os.path.join("data", "books", "*.txt")))
allBooksRDD = sc.textFile(",".join(os.path.abspath(p).replace("\\", "/") for p in book_files))

In [17]:
bookRDD.count()

12760

In [18]:
allBooksRDD.count()

183019

In [19]:
allWordsRDD = allBooksRDD.flatMap(lambda line: line.split())
allWordsRDD.countByValue()

defaultdict(int,
            {'The': 7235,
             'Project': 2265,
             'Gutenberg': 597,
             'EBook': 35,
             'of': 51286,
             'Outline': 9,
             'Science,': 5,
             'Vol.': 12,
             '1': 54,
             '(of': 8,
             '4),': 2,
             'by': 7078,
             'J.': 230,
             'Arthur': 30,
             'Thomson': 7,
             'This': 1029,
             'eBook': 145,
             'is': 11519,
             'for': 10202,
             'the': 86255,
             'use': 607,
             'anyone': 145,
             'anywhere': 71,
             'at': 6970,
             'no': 3012,
             'cost': 115,
             'and': 40408,
             'with': 11142,
             'almost': 568,
             'restrictions': 63,
             'whatsoever.': 61,
             'You': 1014,
             'may': 1928,
             'copy': 258,
             'it,': 1089,
             'give': 750,
             'it': 9818

*Only if you are running this notebook on your local computer, not if running on Google Colab*: Open the Spark UI and click on the last job to view its details. Look at the DAG visualisation and click on it to view the details. You will see several tasks being run in parallel (depending on your setup).

### Other transformations: sample, distinct, filter

In [20]:
#Get a sample of 10% of the words
sample = allWordsRDD.sample(False, 0.1)
sample.count()

151953

In [21]:
allWordsRDD.count()

1517786

In [22]:
#map to lower case, ignore duplicates
lowerSample = sample.map(lambda w: w.lower())
lowerSample.distinct().count()

28382

In [23]:
# Filter allows to get the subset of elements of an RDD that satisfy a given predicate
startsWithARDD = allWordsRDD.filter(lambda w: w.startswith('a'))
print("First three elements starting with \'a\'", startsWithARDD.take(3))
print("Number of elements starting with \'a\'", startsWithARDD.count() )
print("Number of distinct elements starting with \'a\'", startsWithARDD.distinct().count())

First three elements starting with 'a' ['anyone', 'anywhere', 'at']


Number of elements starting with 'a' 147279


Number of distinct elements starting with 'a' 5249


**Exercise**: the code above counts all words that start with lowercase letter 'a'. Using the same filter predicate (i.e., `lambda w: w.startswith('a')`, calculate the number of words that start with either 'a' or 'A'.

In [24]:
# Your answer here

### Default aggregations on RDD of ints/RDD of doubles

In [25]:
# Various easy aggregation function are pre-defined on RDDs containing only integers
intsRDD = sc.parallelize(range(1,10))
intsRDD.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [26]:
intsRDD.sum()

45

In [27]:
intsRDD.mean()

5.0

In [28]:
intsRDD.stdev()

np.float64(2.581988897471611)

In [29]:
intsRDD.variance()

6.666666666666667

In [30]:
intsRDD.stats()

(count: 9, mean: 5.0, stdev: 2.581988897471611, max: 9.0, min: 1.0)

### Set operations

In [31]:
# UNION : Build the union of a list of RDDs; the result will have duplicates

one = sc.parallelize(range(1,10))
two = sc.parallelize(range(9,21))
print("One : ",one.collect())
print("Two : ",two.collect())
print("One union Two : ",one.union(two).collect())

One :  [1, 2, 3, 4, 5, 6, 7, 8, 9]


Two :  [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


One union Two :  [1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [32]:
# INTERSECTION : Return the intersection of this RDD and another one. 
# The output will not contain any duplicate elements, even if the input RDDs did.

one = sc.parallelize(range(1,10))
two = sc.parallelize(range(5,15))
print("One : ",one.collect())
print("Two : ",two.collect())
print("One intersection Two : ",one.intersection(two).collect())

One :  [1, 2, 3, 4, 5, 6, 7, 8, 9]


Two :  [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


One intersection Two :  [8, 5, 9, 6, 7]


In [33]:
# Subtract computes set difference (without duplicates)
one = sc.parallelize(range(1,10))
two = sc.parallelize(range(5,15))
print("One : ",one.collect())
print("Two : ",two.collect())
print("One difference Two : ",one.subtract(two).collect())

One :  [1, 2, 3, 4, 5, 6, 7, 8, 9]


Two :  [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


One difference Two :  [4, 1, 2, 3]


### Partitioning

In [34]:
# An RDD is partitioned transparently

# Using the glom() transformation we can inspect how the elements of an RDD are partitioned.
# glom() coalesces all elements within each partition into a list. 

# Using the repartition() function we can transform an RDD into an RDD  
# that has exactly numPartitions partitions. We Can increase or decrease the level of
# parallelism in this RDD. Internally, this uses a shuffle to redistribute data. 

# We ask the collection to have 5 partitions
rdd = sc.parallelize(range(50), 5).map(str)
glomed = rdd.glom()
print("Initial RDD : ",rdd.collect())
print("Glomed RDD : ",glomed.collect())
print("# of partitions : ",rdd.getNumPartitions())

print("-----------------------------")
rdd = sc.parallelize(range(50), 4)        # We specify 4 partitions
print("# of partitions before ",rdd.getNumPartitions())
print("Data in partitions : ",rdd.collect())
glomed = rdd.glom()
print("Glomed RDD : ",glomed.collect())

print("-----------------------------")
repartitionedRDD = rdd.repartition(10)
print("# of partitions after ",repartitionedRDD.getNumPartitions())
print("Data in partitions : ",repartitionedRDD.collect())
glomed = repartitionedRDD.glom()
print("Glomed RDD : ",glomed.collect())


Initial RDD :  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49']


Glomed RDD :  [['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], ['10', '11', '12', '13', '14', '15', '16', '17', '18', '19'], ['20', '21', '22', '23', '24', '25', '26', '27', '28', '29'], ['30', '31', '32', '33', '34', '35', '36', '37', '38', '39'], ['40', '41', '42', '43', '44', '45', '46', '47', '48', '49']]
# of partitions :  5
-----------------------------
# of partitions before  4


Data in partitions :  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


Glomed RDD :  [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24], [25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36], [37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]]
-----------------------------
# of partitions after  10


Data in partitions :  [10, 11, 35, 36, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 47, 48, 49]


Glomed RDD :  [[10, 11, 35, 36], [], [], [], [12, 13, 14, 15, 16, 17, 18, 19, 20, 21], [22, 23, 24], [], [], [37, 38, 39, 40, 41, 42, 43, 44, 45, 46], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 47, 48, 49]]


### Pair RDDs

In [35]:
# Pair RDDs are RDDs that contain pairs of the form (key, value)
elems = [ ('a', [1,2,3,4]), ('b', [2,3,4]), ('c', [5,9,11,12]), ('a', [100,101])]
pairRDD = sc.parallelize(elems)

In [36]:
pairRDD.take(2)

[('a', [1, 2, 3, 4]), ('b', [2, 3, 4])]

In [37]:
# Pair RDDs have some special operations defined on them

# mapValues: pass each value in the key-value pair RDD through a map function without changing the keys
# this also retains the original RDD’s partitioning.
print("Mapping values to a string: ", pairRDD.mapValues(lambda value: str(value)).collect())
print("Mapping values to take only first element:", pairRDD.mapValues(lambda value: value[0]).collect())
print("Mapping values to sum of elements:", pairRDD.mapValues(lambda value: sum(value)).collect())

Mapping values to a string:  [('a', '[1, 2, 3, 4]'), ('b', '[2, 3, 4]'), ('c', '[5, 9, 11, 12]'), ('a', '[100, 101]')]


Mapping values to take only first element: [('a', 1), ('b', 2), ('c', 5), ('a', 100)]


Mapping values to sum of elements: [('a', 10), ('b', 9), ('c', 37), ('a', 201)]


In [38]:
# We can also group all values that have the same key. Each key then has a value that is a collection
pairRDD.groupByKey().collect()

[('b', <pyspark.resultiterable.ResultIterable at 0x20ece3e9550>),
 ('c', <pyspark.resultiterable.ResultIterable at 0x20ea554f390>),
 ('a', <pyspark.resultiterable.ResultIterable at 0x20ece42cf50>)]

In [39]:
# We can actually do the usual kind of things on these collections
pairRDD.groupByKey().mapValues(lambda value: len(value)).collect()

[('b', 1), ('c', 1), ('a', 2)]

In [40]:
# The following converts the value from a ResultIterable to a normal list
pairRDD.groupByKey().mapValues(lambda value: [x for x in value]).collect()

[('b', [[2, 3, 4]]),
 ('c', [[5, 9, 11, 12]]),
 ('a', [[1, 2, 3, 4], [100, 101]])]

In [41]:
# Reducebykey is a parallel reduce operation, that reduces per key the set of all 
# associated values into a single value
pairRDD.reduceByKey(lambda x,y: x+y).collect()   #Note: + on lists is list concatenation

[('b', [2, 3, 4]), ('c', [5, 9, 11, 12]), ('a', [1, 2, 3, 4, 100, 101])]

In [42]:
# Aggregate by key is similar, but it allows us more flexibility on how to combine
# In the following example, we start with an initial aggregate value 0. Then, 
# for each (key, val) pair in the RDD, val is first aggregated using lambda x, y: x+sum(y)
# Note that in this function, x is the aggregate value so far and y is the key-value (in our case it is a list)
# the new aggregate is obtained by adding the old aggregate value to the sum of the list. 
# Finally, multiple aggregate values for the same key are combined into a single one using lambda x,y: x+y,
# i.e., just by adding the per-pair aggregate values.
pairRDD.aggregateByKey(0, lambda x, y: x+ sum(y), lambda x, y: x + y).collect()

[('b', 9), ('c', 37), ('a', 211)]

In [43]:
# CAUTION!
# Be careful about using pairRDD methods on non-pair RDDs. The result is not always what you expect.
elems = [ [1,2,3,4], [2,3,4,5]]
rdd = sc.parallelize(elems)
print(rdd.collect())
print(rdd.mapValues(lambda x: x).collect())

# Note thbat hence each list is viewed as a pair, where the first element is the key and the second
# element is the value, the rest is discarded

[[1, 2, 3, 4], [2, 3, 4, 5]]


[(1, 2), (2, 3)]


In [44]:
# Pair RDDs can also be sorted, either by key, or by some computed sort key
elems = [ ('a', [1,2,3,4]), ('b', [2,3,4]), ('c', [5,9,11,12]), ('a', [100,101])]
pairRDD = sc.parallelize(elems)
print("sorted by key:", pairRDD.sortByKey().collect())
print("sorted descendingly by length of the value:", pairRDD.sortBy(lambda x: len(x[1]), False).collect())

sorted by key: [('a', [1, 2, 3, 4]), ('a', [100, 101]), ('b', [2, 3, 4]), ('c', [5, 9, 11, 12])]


sorted descendingly by length of the value: [('a', [1, 2, 3, 4]), ('c', [5, 9, 11, 12]), ('b', [2, 3, 4]), ('a', [100, 101])]


In [45]:
# Normal RDDs can be transformed into pair RDDs by the groupBy transformation
# GroupBy takes a function f that returns, for each element in the old RDD, its key in the new rdd
# All elements with the same key are grouped together in a value
elems = ['abcd', 'abracadabra', 'hello', 'hi', 'bottom', 'top']
rdd = sc.parallelize(elems)
print("grouped by first letter:\n", str(rdd.groupBy(lambda x: x[0]).collect()))
print("grouped by first letter, values mapped to lists:\n", rdd.groupBy(lambda x: x[0]).mapValues(lambda x: [y for y in x]).collect())    

grouped by first letter:
 [('b', <pyspark.resultiterable.ResultIterable object at 0x0000020ECE4D42B0>), ('t', <pyspark.resultiterable.ResultIterable object at 0x0000020ECE4D4510>), ('a', <pyspark.resultiterable.ResultIterable object at 0x0000020EA556BD10>), ('h', <pyspark.resultiterable.ResultIterable object at 0x0000020ECE416CF0>)]


grouped by first letter, values mapped to lists:
 [('b', ['bottom']), ('t', ['top']), ('a', ['abcd', 'abracadabra']), ('h', ['hello', 'hi'])]


In [46]:
# Pair RDDs can be joined based on their key
elems1 = [ ('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5)]
elems2 = [ ('a', 'a1'), ('b', 'b1'), ('d', 'd1')]
rdd1 = sc.parallelize(elems1)
rdd2 = sc.parallelize(elems2)
rdd1.join(rdd2).collect()

[('d', (4, 'd1')), ('b', (2, 'b1')), ('a', (1, 'a1'))]

In [47]:
# Outer joins keep the keys that do not have a matching key on the other side
rdd1.leftOuterJoin(rdd2).collect()

[('d', (4, 'd1')),
 ('e', (5, None)),
 ('b', (2, 'b1')),
 ('c', (3, None)),
 ('a', (1, 'a1'))]

## Cleaning up

It is **vital** that you stop your spark instance after you are done working with it, **especially when running in cluster mode**. If you do not do this, the resources acquired by your spark instance (i.e., the number of cores and memory reserved for it) will be kept indefinetely, and *are hence not available for others!*

In [48]:
sc.stop()